# Nature-Mirror Mesh — Free Colab Lab ($0)

**What this is:** the Nature-Mirror Mesh lab proofs, packaged to run end-to-end on a
**free Google Colab** runtime. No GPU is needed (CPU runtime is fine) and nothing costs anything.

**How to run:** open this notebook in Google Colab, then **Runtime → Run all**.
Total time ≈ 5 minutes. Each step prints its own PASS/FAIL.

You will run:
1. **FT8 weak-signal round-trip** — build the open-source `ft8_lib` C library from source,
   encode a message into a WAV, bury it in noise (down to −15 dB), and decode it back.
2. **LXMF two-node encrypted chat test** — install Reticulum (`rns`) + `lxmf` and deliver an
   end-to-end encrypted message between two independent nodes, exactly like the lab proof.

> Alternative for step 1: Colab can also `!apt-get install -y wsjtx`, which provides the `jt9`
> FT8 decoder plus sample WAVs. This notebook uses `ft8_lib` because it is small and builds in seconds.


## Step 1 — FT8: build ft8_lib, encode → noise → decode

In [ ]:
import subprocess, os
def run(cmd, **kw):
    print("+", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(r.stdout[-3000:]); print(r.stderr[-1000:])
    return r
os.chdir('/content')
run("rm -rf ft8_lib && git clone --depth 1 https://github.com/kgoba/ft8_lib")
os.chdir('/content/ft8_lib')
run("make")
r = run('./gen_ft8 "CQ NMM PK" /tmp/ft8_clean.wav 1000')
print(r.stdout)


In [ ]:
import numpy as np, wave, subprocess, os
os.chdir('/content/ft8_lib')
w = wave.open('/tmp/ft8_clean.wav', 'rb')
n, fr = w.getnframes(), w.getframerate()
x = np.frombuffer(w.readframes(n), dtype=np.int16).astype(np.float64); w.close()
sig = np.sqrt(np.mean(x**2)); rng = np.random.default_rng(42)
for snr in (-10, -15):
    y = np.clip(x + rng.normal(0, sig / (10 ** (snr / 20)), len(x)), -32767, 32767).astype(np.int16)
    fn = f"/tmp/ft8_snr{snr}.wav"
    wo = wave.open(fn, 'wb'); wo.setnchannels(1); wo.setsampwidth(2); wo.setframerate(fr)
    wo.writeframes(y.tobytes()); wo.close()
    out = subprocess.run(['./decode_ft8', fn], capture_output=True, text=True).stdout
    print(f"--- full-band RMS SNR {snr} dB ---"); print(out)
    assert "CQ NMM PK" in out, f"decode failed at {snr} dB"
print("FT8 ROUND-TRIP: PASS")


## Step 2 — Reticulum + LXMF: two-node encrypted delivery

Installs `rns` and `lxmf`, writes a self-contained two-node test (two separate Reticulum
processes linked over local TCP), and runs it. PASS = the receiver prints the exact
unique token the sender encrypted for it.

In [ ]:
import subprocess
r = subprocess.run("pip install -q rns lxmf", shell=True, capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
print("install done")


In [ ]:
LXMF_SRC = '#!/usr/bin/env python3\n"""Compact two-node LXMF test for Google Colab (adapted from lab/lxmf_two_node_test.py).\nTwo Reticulum processes on 127.0.0.1 TCP; exit 0 only on exact token match."""\nimport os, sys, time, subprocess, shutil\n\nWORK = "/tmp/nmm_colab_work"\nPORT = 51811\nTOKEN_F = os.path.join(WORK, "token.txt")\nDEST_F = os.path.join(WORK, "dest.txt")\nRECV_F = os.path.join(WORK, "received.txt")\n\ndef cfg(configdir, server):\n    os.makedirs(configdir, exist_ok=True)\n    iface = ("  [[L]]\\n    type = TCPServerInterface\\n    enabled = True\\n"\n             f"    listen_ip = 127.0.0.1\\n    listen_port = {PORT}") if server else \\\n            ("  [[L]]\\n    type = TCPClientInterface\\n    enabled = True\\n"\n             f"    target_host = 127.0.0.1\\n    target_port = {PORT}")\n    open(os.path.join(configdir, "config"), "w").write(\n        "[reticulum]\\n  enable_transport = False\\n  share_instance = False\\n\\n"\n        "[logging]\\n  loglevel = 4\\n\\n[interfaces]\\n" + iface + "\\n")\n\ndef receiver():\n    import RNS, LXMF\n    cfg(os.path.join(WORK, "a"), True)\n    RNS.Reticulum(configdir=os.path.join(WORK, "a"))\n    ident = RNS.Identity()\n    router = LXMF.LXMRouter(identity=ident, storagepath=os.path.join(WORK, "sa"))\n    dest = router.register_delivery_identity(ident, display_name="Colab A", stamp_cost=None)\n    def on_msg(m):\n        c = m.content_as_string()\n        print(f"[A] DECRYPTED: {c}", flush=True)\n        open(RECV_F, "w").write(c)\n    router.register_delivery_callback(on_msg)\n    open(DEST_F, "w").write(RNS.hexrep(dest.hash, delimit=False))\n    end = time.time() + 120\n    while time.time() < end and not os.path.exists(RECV_F):\n        router.announce(dest.hash); time.sleep(4)\n    sys.exit(0 if os.path.exists(RECV_F) else 2)\n\ndef sender():\n    import RNS, LXMF\n    cfg(os.path.join(WORK, "b"), False)\n    RNS.Reticulum(configdir=os.path.join(WORK, "b"))\n    token = open(TOKEN_F).read().strip()\n    dh = bytes.fromhex(open(DEST_F).read().strip())\n    ident = RNS.Identity()\n    router = LXMF.LXMRouter(identity=ident, storagepath=os.path.join(WORK, "sb"))\n    src = router.register_delivery_identity(ident, display_name="Colab B", stamp_cost=None)\n    end = time.time() + 60\n    while not RNS.Transport.has_path(dh):\n        RNS.Transport.request_path(dh)\n        if time.time() > end: sys.exit(2)\n        time.sleep(0.5)\n    di = RNS.Identity.recall(dh)\n    dest = RNS.Destination(di, RNS.Destination.OUT, RNS.Destination.SINGLE, "lxmf", "delivery")\n    msg = LXMF.LXMessage(dest, src, content=token, title="colab-test")\n    router.handle_outbound(msg)\n    print(f"[B] sent token={token}", flush=True)\n    end = time.time() + 45\n    while time.time() < end and msg.state != LXMF.LXMessage.DELIVERED:\n        time.sleep(0.5)\n    print(f"[B] final state={msg.state}", flush=True)\n    sys.exit(0)\n\ndef main():\n    shutil.rmtree(WORK, ignore_errors=True); os.makedirs(WORK)\n    token = "NMM-COLAB-" + time.strftime("%Y%m%d-%H%M%S")\n    open(TOKEN_F, "w").write(token)\n    print("Token:", token, flush=True)\n    py, me = sys.executable, os.path.abspath(__file__)\n    ra = subprocess.Popen([py, me, "receiver"])\n    end = time.time() + 30\n    while not os.path.exists(DEST_F):\n        if time.time() > end: print("FAIL: no dest"); ra.kill(); return 1\n        time.sleep(0.25)\n    sb = subprocess.Popen([py, me, "sender"])\n    sb.wait(timeout=150)\n    try: ra.wait(timeout=20)\n    except subprocess.TimeoutExpired: ra.terminate()\n    if os.path.exists(RECV_F) and open(RECV_F).read() == token:\n        print("RESULT: PASS — LXMF encrypted delivery verified (Colab two-node test)")\n        return 0\n    print("RESULT: FAIL"); return 1\n\nif __name__ == "__main__":\n    if len(sys.argv) > 1 and sys.argv[1] == "receiver": receiver()\n    elif len(sys.argv) > 1 and sys.argv[1] == "sender": sender()\n    else: sys.exit(main())\n'

with open('/tmp/colab_lxmf_test.py', 'w') as f:
    f.write(LXMF_SRC)
print("test file written:", len(LXMF_SRC), "chars")


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '/tmp/colab_lxmf_test.py'], capture_output=True, text=True, timeout=300)
print(r.stdout); print(r.stderr[-1000:])
assert r.returncode == 0, "LXMF test FAILED"
print("COLAB LAB: ALL STEPS PASS")


## Honesty notes

- These are **software proofs**: the encrypted transport layer (Reticulum/LXMF) and the FT8
  weak-signal codec. No radio is transmitted from Colab.
- Next physical step (needs any ordinary machine + a $30 RTL-SDR, or a public KiwiSDR):
  record real off-air FT8 audio and feed the WAV to `decode_ft8` — the decoder side is
  already proven by Step 1.
- Project: Nature-Mirror Mesh — a free, offline, end-to-end-encrypted chat backend that
  borrows nature's mirrors (ionosphere, meteor trails, the Moon) instead of rented infrastructure.
